[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/17_save_load_deploy.ipynb)

# Save, load & run inference (deploy)

> **Fine-tuning series — 17 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## Save, load & run inference (deploy)

You trained adapters — now save them, reload them, run generation, and optionally merge or
export for serving. This notebook trains a quick adapter first so there's something to save.

### Train a quick adapter (so we have something to save)

In [ ]:
import torch
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
trainer = SFTTrainer(model=model, train_dataset=sft_ds, peft_config=lora_cfg,
    args=SFTConfig(output_dir="demo", dataset_text_field="text",
                   per_device_train_batch_size=2, max_steps=20, learning_rate=2e-4,
                   logging_steps=10, bf16=BF16_OK, report_to="none"))
trainer.train()

### Save the adapter

`save_model` writes **only the adapter** (a few MB) plus its config — not the whole base model.
Save the tokenizer alongside it.

In [ ]:
trainer.save_model("my_adapter")          # adapter weights + adapter_config.json
tokenizer.save_pretrained("my_adapter")
import os; print("saved:", os.listdir("my_adapter"))
del trainer, model; cleanup()

### Reload for inference (adapter on top of base)

Load the base model, attach the saved adapter with `PeftModel.from_pretrained`, and generate.
Keeping the adapter separate lets you hot-swap many adapters on one base.

In [ ]:
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
infer = PeftModel.from_pretrained(base, "my_adapter").eval()

def generate(prompt, max_new_tokens=64, temperature=0.0):
    # return_dict=True -> splat into generate(); transformers 5.x no longer returns a
    # bare tensor from apply_chat_template(return_tensors="pt").
    enc = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
            add_generation_prompt=True, return_tensors="pt", return_dict=True).to(infer.device)
    out = infer.generate(**enc, max_new_tokens=max_new_tokens,
                         do_sample=temperature > 0, temperature=max(temperature, 1e-5))
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

print(generate("Say hello in one word."))

### Merge for zero-overhead serving (optional)

Fold the adapter into the weights so there's no runtime overhead. **QLoRA caveat:** reload the
base in **bf16 (not 4-bit)** before merging — you can't meaningfully merge into a 4-bit base.

In [ ]:
merged = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": 0}),
    "my_adapter").merge_and_unload()
merged.save_pretrained("my_merged_model")
tokenizer.save_pretrained("my_merged_model")
print("merged full model saved -> my_merged_model")

### Export / serve options

- **Hugging Face Hub:** `model.push_to_hub("user/name")` + `tokenizer.push_to_hub("user/name")`.
- **GGUF (llama.cpp / Ollama):** convert the merged model with llama.cpp's `convert_hf_to_gguf.py`,
  or in Unsloth `model.save_pretrained_gguf("out", tokenizer, quantization_method="q4_k_m")`,
  then `ollama create <name> -f Modelfile`.
- **High-throughput serving:** load the merged model (or base + adapter) in **vLLM** or **TGI**.

### Generation parameters (quick reference)

- `do_sample=False` → greedy (deterministic); `do_sample=True` + `temperature` → varied.
- `temperature` low (0–0.3) = focused/factual; high (0.7–1.0) = creative.
- `top_p` (e.g. 0.9) = nucleus sampling; `max_new_tokens` caps output length.